In [1]:
%pip install openai scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",  # Ollama
    api_key="sk-1234"  # Dummy key
)

In [ ]:
SYSTEM_PROMPT = """
You are a DASS-21 survey scoring assistant.
Your job is to extract answers for the DASS-21 based on a given conversation transcript.
The Depression Anxiety Stress Scales 21-item version (DASS-21) is a clinical assessment tool that measures the severity of depression, anxiety, and stress over the past week.

The conversation asks questions in this specific order. Map each user response to the correct q number:
q1  (Stress):      Hard to wind down or switch off
q6  (Stress):      Over-reacting to situations
q8  (Stress):      Using a lot of nervous energy
q11 (Stress):      Getting agitated
q12 (Stress):      Difficult to relax
q14 (Stress):      Impatient or intolerant
q18 (Stress):      Touchy or irritable
q2  (Anxiety):     Dryness of mouth
q4  (Anxiety):     Difficulty breathing
q7  (Anxiety):     Trembling
q9  (Anxiety):     Worried about panicking or embarrassing self
q15 (Anxiety):     Close to panic
q19 (Anxiety):     Aware of heart beating without physical exertion
q20 (Anxiety):     Scared without a clear reason
q3  (Depression):  Couldn't experience any positive feeling
q5  (Depression):  Difficult to work up initiative
q10 (Depression):  Nothing to look forward to
q13 (Depression):  Down-hearted or blue
q16 (Depression):  Unable to become enthusiastic
q17 (Depression):  Not worth much as a person
q21 (Depression):  Life felt meaningless

Scale mapping:
0 = never
1 = sometimes
2 = often
3 = almost always

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value: "0", "1", "2", or "3".
Rules:
- Read each user response in order. The first user response answers q1, the second answers q6, the third answers q8, and so on following the order above.
- Score each response based solely on what the user said.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q21 key.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q6": "0",
  "q8": "0",
  "q11": "0",
  "q12": "0",
  "q14": "0",
  "q18": "0",
  "q2": "0",
  "q4": "0",
  "q7": "0",
  "q9": "0",
  "q15": "0",
  "q19": "0",
  "q20": "0",
  "q3": "0",
  "q5": "0",
  "q10": "0",
  "q13": "0",
  "q16": "0",
  "q17": "0",
  "q21": "0"
}
"""
MODEL = 'qwen3:8b'

- Use a json dataset of DASS-21 conversation transcripts
- Extract scores using LLM
- Validate whether they were correct using MSE

In [4]:
# Load the DASS-21 dataset
import json

with open('dass_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')
print(f'Total conversations: {len(conversations)}')

SUCCESS: Number of conversations matches number of scores
Total conversations: 10


In [5]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(f'Train: {len(conv_train)}, Test: {len(_conv_test)}')

Train: 7, Test: 3


In [6]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(transcript) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        temperature=0.1,  # low temperature for consistent scoring
        messages=[
            Message(SYSTEM_PROMPT, 'system'),
            Message(json.dumps(transcript))
        ],
    )
    return completion.choices[0].message.content

In [7]:
# Run predictions on training set
scores = []
for i, c in enumerate(conv_train):
    print(f'Generating {i+1}/{len(conv_train)}')
    scores.append(create_prediction(c))
    print(scores[-1])

Generating 1/7
{
  "q1": "3",
  "q2": "2",
  "q3": "1",
  "q4": "0",
  "q5": "1",
  "q6": "2",
  "q7": "1",
  "q8": "2",
  "q9": "0",
  "q10": "1",
  "q11": "2",
  "q12": "2",
  "q13": "1",
  "q14": "2",
  "q15": "0",
  "q16": "0",
  "q17": "1",
  "q18": "1",
  "q19": "1",
  "q20": "0",
  "q21": "0"
}
Generating 2/7
{
  "q1": "1",
  "q2": "1",
  "q3": "0",
  "q4": "0",
  "q5": "1",
  "q6": "0",
  "q7": "0",
  "q8": "1",
  "q9": "0",
  "q10": "0",
  "q11": "0",
  "q12": "1",
  "q13": "0",
  "q14": "1",
  "q15": "0",
  "q16": "0",
  "q17": "1",
  "q18": "0",
  "q19": "0",
  "q20": "1",
  "q21": "0"
}
Generating 3/7
{
  "q1": "2",
  "q2": "2",
  "q3": "2",
  "q4": "0",
  "q5": "1",
  "q6": "1",
  "q7": "1",
  "q8": "2",
  "q9": "1",
  "q10": "2",
  "q11": "2",
  "q12": "2",
  "q13": "1",
  "q14": "1",
  "q15": "1",
  "q16": "1",
  "q17": "1",
  "q18": "1",
  "q19": "1",
  "q20": "1",
  "q21": "1"
}
Generating 4/7
{
  "q1": "3",
  "q2": "3",
  "q3": "2",
  "q4": "1",
  "q5": "2",
  "q6": "

In [8]:
# Parse JSON scores
json_scores = []
for s in scores:
    try:
        json_scores.append(json.loads(s))
    except json.JSONDecodeError:
        print(f'Failed to parse: {s}')
        json_scores.append({f'q{i}': '0' for i in range(1, 22)})

In [9]:
from sklearn.metrics import mean_squared_error

def convert_scores_to_array(scores_df):
    return [int(s) if s else 0 for s in scores_df.values.flatten().tolist()]

pred_df = pd.DataFrame(json_scores)
train_df = pd.DataFrame(score_train)

# Ensure column order matches
cols = [f'q{i}' for i in range(1, 22)]
pred_df = pred_df[cols]
train_df = train_df[cols]

mse = mean_squared_error(
    convert_scores_to_array(train_df),
    convert_scores_to_array(pred_df)
)
print(f'MSE: {mse}')

diff_df = train_df.compare(pred_df, align_axis=1, keep_shape=True, keep_equal=True).rename(
    columns={'self': 'Expected', 'other': 'Predicted'}, level=1
)
display(diff_df)

MSE: 0.7142857142857143


q1                 q2                 q3                 q4            \
  Expected Predicted Expected Predicted Expected Predicted Expected Predicted   
0        3         3        2         2        1         1        2         0   
1        1         1        1         1        0         0        0         0   
2        2         2        2         2        2         2        2         0   
3        3         3        3         3        2         2        3         1   
4        1         1        1         1        1         1        1         0   
5        2         2        2         2        0         0        2         1   
6        2         2        1         1        0         3        1         0   

        q5            ...      q17                q18                q19  \
  Expected Predicted  ... Expected Predicted Expected Predicted Expected   
0        1         1  ...        1         1        1         1        0   
1        0         1  ...        0         1        1         0        0   
2        1         1  ...        1         1        2         1        0   
3        2         2  ...        2         2        2         1        2   
4        1         1  ...        1         1        2         1        1   
5        1         1  ...        0         1        2         1        0   
6        2         3  ...        3         3        0         0        0   

                 q20                q21            
  Predicted Expected Predicted Expected Predicted  
0         1        0         0        0         0  
1         0        1         1        0         0  
2         1        1         1        1         1  
3         1        2         2        2         2  
4         1        1         1        0         0  
5         1        0         1        0         0  
6         1        0         1        2         2  

[7 rows x 42 columns]